In [1]:
!pip install boto3  unsloth wandb torch

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
key = user_secrets.get_secret("AWS_ACCESS_KEY_ID")
secret = user_secrets.get_secret("AWS_SECRET_ACCESS_KEY")
wandb= user_secrets.get_secret("wandb")
hf_key=user_secrets.get_secret("hf_key")

In [4]:
import os
os.environ["AWS_ACCESS_KEY_ID"] = key
os.environ["AWS_SECRET_ACCESS_KEY"] = secret
os.environ["WANDB_API_KEY"]= wandb

In [ ]:
from unsloth import FastLanguageModel
import wandb
from peft import PeftModel
import torch
import tqdm
import boto3
from botocore.config import Config
from datasets import load_dataset

class Model:

    def __init__(self):
        self.tokenizer = None

    def dataset_load(self):
        s3 = boto3.client(
            "s3",
            endpoint_url="https://s3.hf.co/Rakesh7n",
            config=Config(
                region_name="us-east-1",
                s3={"addressing_style": "path"},
                request_checksum_calculation="when_required",
                response_checksum_validation="when_required",
            ),
        )
        s3.download_file("ncrt", "transformed_dataset", "test.csv")
        ds = load_dataset("csv", data_files="test.csv")
        return ds["train"]  

    def generate_batch(self, model, prompts, max_new_tokens=512):
        inputs = self.tokenizer(prompts, return_tensors="pt", padding=True, truncation=True)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0)
        generated_texts = []
        for i, output in enumerate(outputs):
            input_len = inputs["input_ids"][i].shape[0]
            gen_text = self.tokenizer.decode(output[input_len:], skip_special_tokens=True)
            generated_texts.append(gen_text)
        return generated_texts

    def model_and_adapter_response(self, batch_size: int = 64):
        ds = self.dataset_load()

        # Download adapter from wandb
        api = wandb.Api()
        artifact = api.artifact("rakeshthalaboina11-jb-institute-of-engineering-and-techn/Qwen3_4B_NCRT_Physics_12th_Finetuned/model-u89p9pjz:v4")
        adapter_dir = artifact.download()

        # Load base model 4-bit
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name="unsloth/Qwen3-4B-Thinking-2507",
            max_seq_length=2048,
            load_in_4bit=True,
        )
        self.tokenizer = tokenizer
        tokenizer.pad_token = tokenizer.eos_token

        prompts = ds["formatted_prompt"]

        # Base model eval
        FastLanguageModel.for_inference(model)
        model.eval()

        base_outputs = []
        for i in tqdm.tqdm(range(0, len(prompts), batch_size)):
            batch_prompts = prompts[i : i + batch_size]
            base_outputs.extend(self.generate_batch(model, batch_prompts))

        #Adapter eval
        adapter_model = PeftModel.from_pretrained(model, adapter_dir)
        adapter_model.eval()

        adapter_outputs = []
        for i in tqdm.tqdm(range(0, len(prompts), batch_size)):
            batch_prompts = prompts[i : i + batch_size]
            adapter_outputs.extend(self.generate_batch(adapter_model, batch_prompts))

        del adapter_model
        del model
        torch.cuda.empty_cache()

        ds = ds.add_column("base_output", base_outputs)
        ds = ds.add_column("adapter_output", adapter_outputs)
        

        return ds   

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1635: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [7]:
model=Model()
ds=model.model_and_adapter_response()

Generating train split: 0 examples [00:00, ? examples/s]

wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Downloading large artifact 'model-u89p9pjz:v4', 267.27MB. 12 files...
wandb:   12 of 12 files downloaded.  
Done. 00:00:00.2 (1382.4MB/s)


==((====))==  Unsloth 2026.10.3: Fast Qwen3 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Unsloth: Not splitting across GPUs; the model fits on cuda:0 with room to train. Pass device_map = "unsloth_balanced" to split it anyway.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

100%|██████████| 9/9 [1:01:27<00:00, 409.71s/it]


In [8]:
from huggingface_hub import login
login(token=hf_key)


In [9]:

ds.push_to_hub(
    "Rakesh7n/ncrt_eval_outputs",
    private=True,          
    commit_message="base + adapter outputs",
)   

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.
[datasets.arrow_dataset|WARNING]Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/Rakesh7n/ncrt_eval_outputs/commit/a56885b0377fa3ee9145d0b892d8d375ec4f86a2', commit_message='base + adapter outputs', commit_description='', oid='a56885b0377fa3ee9145d0b892d8d375ec4f86a2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Rakesh7n/ncrt_eval_outputs', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Rakesh7n/ncrt_eval_outputs'), pr_revision=None, pr_num=None)